In [1]:
import gzip
from pathlib import Path
import pandas as pd

path = Path("GSE62254_series_matrix.txt")   # change to your file name
opener = gzip.open if path.suffix == ".gz" else open

ids, titles, rows = [], [], None
with opener(path, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("!series_matrix_table_begin"):
            break                      # the expression table starts here
        parts = line.rstrip("\n").split("\t")
        tag, vals = parts[0], [v.strip('"') for v in parts[1:]]
        if tag == "!Sample_title":
            titles = vals
        elif tag == "!Sample_geo_accession":
            ids = vals
            rows = [dict() for _ in ids]
        elif tag == "!Sample_characteristics_ch1" and rows is not None:
            for i, v in enumerate(vals):
                if ":" in v:
                    key, val = v.split(":", 1)
                    rows[i][key.strip()] = val.strip()

df = pd.DataFrame(rows, index=ids)
if len(titles) == len(df):
    df.insert(0, "title", titles)
df = df.replace({"NA": pd.NA, "N/A": pd.NA, "n/a": pd.NA, "": pd.NA})

print(df.shape)
print()
for c in df.columns:
    print(f"{c:35s} distinct values: {df[c].nunique():4d}   missing: {df[c].isna().sum():4d}")
print()
print(df.head())
df.to_csv("acrg_clinical.csv")

(300, 3)

title                               distinct values:  300   missing:    0
tissue                              distinct values:    1   missing:    0
patient                             distinct values:  300   missing:    0

           title         tissue patient
GSM1523727  T107  Gastric tumor     107
GSM1523728  T108  Gastric tumor     108
GSM1523729  T109  Gastric tumor     109
GSM1523744  T160  Gastric tumor     160
GSM1523745  T164  Gastric tumor     164


In [2]:
import re
import pandas as pd

CLIN_XLS = "41591_2015_BFnm3850_MOESM34_ESM.xls"
GEO_IDS  = "acrg_clinical.csv"
OUT      = "acrg_clinical_full.csv"


clin = pd.read_excel(CLIN_XLS, sheet_name="FINAL", engine="xlrd")
clin.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in clin.columns]

RULES = [
    ("eq", "Tumor ID", "tumor_id"), ("eq", "SCRI No.", "scri_no"),
    ("eq", "MLH1 IHC", "mlh1_ihc"), ("eq", "Sample Name", "sample_name"),
    ("eq", "sex", "sex"), ("eq", "age", "age"), ("eq", "Pathology", "pathology"),
    ("sw", "EBV ISH", "ebv_ish"), ("sw", "WHO 1=", "who_code"),
    ("eq", "WHO_simple", "who_simple"), ("sw", "LAUREN 1=", "lauren_code"),
    ("sw", "Perineural", "perineural_invasion"), ("sw", "VENOUS", "venous_invasion"),
    ("sw", "lymphatic", "lymphovascular_invasion"), ("eq", "tumor type", "tumor_type"),
    ("eq", "OP date", "op_date"), ("eq", "stage(TNM)", "stage_tnm"),
    ("eq", "T", "t_raw"), ("eq", "N", "n_raw"), ("eq", "M", "m_raw"),
    ("eq", "pStage", "p_stage_raw"), ("eq", "Lauren", "lauren_raw"),
    ("eq", "pnode", "nodes_examined"), ("sw", "# of positive node", "positive_nodes"),
    ("eq", "revised location", "location"), ("eq", "Code_site", "location_code"),
    ("sw", "OP NAME", "op_type"), ("sw", "EXTENT OF OP", "extent_of_op"),
    ("sw", "Adjuvant CCRT0", "adj_ccrt"), ("sw", "Adjuvant CCRT_C", "adj_ccrt_c"),
    ("sw", "Adjuvant CCRT CYCLE", "adj_ccrt_cycles"), ("sw", "ADJ CTx.", "adj_ctx_description"),
    ("sw", "documented recurrence", "recurrence_documented"),
    ("sw", "Date of recurrence", "recurrence_date"),
    ("sw", "First site of recurrence_liver", "rec_site_liver"),
    ("sw", "First site of recurrence_peritoneal", "rec_site_peritoneal"),
    ("sw", "First site of recurrence_ascites", "rec_site_ascites"),
    ("sw", "First site of recurrence_intraabdominal", "rec_site_intraabd_ln"),
    ("sw", "First site of recurrence_distant", "rec_site_distant_ln"),
    ("sw", "First site of recurrence_bone", "rec_site_bone"),
    ("sw", "recurrence sites_others", "rec_site_others"),
    ("eq", "description", "notes"), ("sw", "FU status", "fu_status"),
    ("sw", "date of death", "last_followup_or_death_date"),
    ("sw", "H. pylori", "h_pylori"), ("sw", "DFS", "dfs_months"),
    ("sw", "OS", "os_months"), ("sw", "Mol. Subtype", "molecular_subtype"),
]

def new_name(col):
    for how, key, new in RULES:
        if (how == "eq" and col == key) or (how == "sw" and col.startswith(key)):
            return new
    return None

mapping = {c: new_name(c) for c in clin.columns}
unmapped = [c for c, n in mapping.items() if n is None]
assert not unmapped, f"Unmapped columns: {unmapped}"
assert len(set(mapping.values())) == len(mapping), "Duplicate new names"
df = clin.rename(columns=mapping)

num = lambda s: pd.to_numeric(s, errors="coerce")
df["tumor_id"] = num(df["tumor_id"]).astype(int)
for c in ["age", "nodes_examined", "positive_nodes", "dfs_months", "os_months"]:
    df[c] = num(df[c])
df["t_stage"], df["n_stage"], df["m_stage"] = num(df["t_raw"]), num(df["n_raw"]), num(df["m_raw"])
df["p_stage"] = df["p_stage_raw"].where(df["p_stage_raw"].isin(["I", "II", "III", "IV"]))
df["lauren"] = df["lauren_raw"].where(df["lauren_raw"].isin(["intestinal", "diffuse", "mixed"]))

df["multiple_tumors"] = df[["t_raw", "p_stage_raw", "lauren_raw"]].astype(str).apply(
    lambda col: col.str.contains("①")).any(axis=1)

rec = num(df["recurrence_documented"])
fu = num(df["fu_status"])
df["recurrence_event"] = rec.map({0: 0, 1: 1})   # 2 (unknown) becomes missing
df["use_for_recurrence"] = df["recurrence_event"].notna()
df["outcome_inconsistent"] = ((rec == 0) & (fu == 3)) | ((rec == 1) & (fu == 0))

ids = pd.read_csv(GEO_IDS, index_col=0).reset_index()
ids = ids.rename(columns={ids.columns[0]: "gsm", "title": "geo_title"}).drop(columns="tissue")
out = ids.merge(df, left_on="patient", right_on="tumor_id", how="left", validate="one_to_one")
assert out["tumor_id"].notna().all(), "Some GEO patients have no clinical row"

first = ["gsm", "geo_title", "patient", "tumor_id", "sex", "age", "t_stage", "n_stage", "m_stage",
         "p_stage", "lauren", "location", "positive_nodes", "nodes_examined",
         "recurrence_event", "dfs_months", "os_months", "use_for_recurrence",
         "outcome_inconsistent", "multiple_tumors"]
out = out[first + [c for c in out.columns if c not in first]]
out.to_csv(OUT, index=False, encoding="utf-8-sig")

print(out.shape)
print(out["recurrence_event"].value_counts(dropna=False))
print("patients usable for recurrence:", int(out["use_for_recurrence"].sum()),
      "| events:", int(out["recurrence_event"].sum()))
print("flagged inconsistent:", int(out["outcome_inconsistent"].sum()),
      "| multiple tumours:", int(out["multiple_tumors"].sum()))
print(out[first].head())

(300, 60)
recurrence_event
0.0    157
1.0    125
NaN     18
Name: count, dtype: int64
patients usable for recurrence: 282 | events: 125
flagged inconsistent: 4 | multiple tumours: 3
          gsm geo_title  patient  tumor_id sex  age  t_stage  n_stage  \
0  GSM1523727      T107      107       107   M   67      2.0        1   
1  GSM1523728      T108      108       108   F   68      2.0        1   
2  GSM1523729      T109      109       109   F   42      2.0        1   
3  GSM1523744      T160      160       160   M   69      2.0        1   
4  GSM1523745      T164      164       164   M   68      3.0        2   

   m_stage p_stage      lauren location  positive_nodes  nodes_examined  \
0        0      II  intestinal     body               3              20   
1        0      II  intestinal     body               1              40   
2        0      II     diffuse   antrum               1              21   
3        0      II     diffuse   antrum               3              24   
4   

C:\Users\patri\AppData\Local\Temp\ipykernel_18796\2147089826.py:9: Pandas4Warning: The xlrd engine is deprecated and will be removed in a future version. Use the calamine engine instead, by setting engine='calamine'.
  clin = pd.read_excel(CLIN_XLS, sheet_name="FINAL", engine="xlrd")
